# 1st Project Trip Advisor - Mantra OUTTANDY, Minji PARK

We will start by looking on our given dataset.

First we will look for reviews data.

In [5]:
import pandas as pd

reviews = pd.read_csv('reviews83325.csv')
places = pd.read_csv('Tripadvisor.csv')

print(reviews.head())
print(reviews.columns)
print(reviews['langue'].value_counts())  # check language distribution

C:\Users\olivi\AppData\Local\Temp\ipykernel_23256\529477921.py:3: DtypeWarning: Columns (15,16,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  reviews = pd.read_csv('reviews83325.csv')


          id  idplace                                           titre  \
0  771569620   188467                                        February   
1  769814072   188467                               Nice green square   
2  758953508   188467                             A Treasure in Paris   
3  755705747   188467  A place to take a breath from exploring Paris.   
4  750396525   188467                           Beautiful and elegant   

                           idauteur  \
0  F645CC9429E8A40EB1F5A487780EC683   
1  AFFB511F21DF819776CB2F8013034382   
2  9262311F3378F8CC4709DD4D92380278   
3  836BAA8786B81033412B950CF5BDA70C   
4  8FEC7B1C9034428EFF3BF8728B036829   

                                              review  note  \
0  Personally I think it is the most beautiful sq...     5   
1  We walked through this lovely park but did not...     4   
2  We come back to this huge square every time we...     5   
3  The most beautiful square in Paris has a stran...     5   
4  Lovely archit

Now we will look for the places('Tripadvisor.csv').

In [9]:
print(places.shape)

(3761, 60)


3761 places available according to our dataset.

In [10]:
print(places.columns.tolist())
print(places.head(2))
print(places['typeR'].value_counts())    # H, R, A, AP distribution

['id', 'idTrip', 'fromId', 'nom', 'url', 'rating', 'nbAvis', 'nbAvisRecupere', 'latitude', 'longitude', 'typeR', 'adresse', 'priceRange', 'closed', 'hotelType', 'hotelStyle', 'hotelStars', 'hotelRoomNumber', 'hotelNoteEmplacement', 'hotelNoteProprete', 'hotelNoteService', 'HotelNoteQualitePrix', 'hoteldistance', 'hotelbearing', 'restaurantTypeCuisine', 'restaurantDietaryRestrictions', 'restaurantMeals', 'restaurantFeatures', 'restaurantNoteCuisine', 'restaurantNoteService', 'restaurantNoteQualitePrix', 'restaurantNoteAmbiance', 'activiteType', 'activiteSubCategorie', 'activiteSubType', 'website', 'nbScanReview', 'dateLastScanReviews', 'shape_gid', 'gadm36_gid', 'hotelprice', 'hotelBookingID', 'restaurantSubcategory', 'restaurantType', 'ap_additional_info', 'ap_age_band_list', 'ap_attraction_ids', 'ap_booking_question_list', 'ap_bubble_rating_integer', 'ap_duration', 'ap_exclusion', 'ap_inclusions', 'ap_introduction', 'ap_primary_supplier_attraction_id', 'ap_primary_supplier_subtype', '

623 Restaurants, 405 Attractions and 64 Hotels are available on the dataset.

In [7]:
reviews_en = reviews[reviews['langue'] == 'en']
reviews_per_place = reviews_en.groupby('idplace').size()
print(reviews_per_place.describe())

count     1835.000000
mean        83.417439
std        828.876539
min          1.000000
25%          2.000000
50%          8.000000
75%         36.000000
max      33878.000000
dtype: float64


We are looking for the reviews written only in english and describing the reviews written by places. There is over 1835 reviews in total written in english. Median only 8 reviews per place. Minimun 1 and maximum 33878 reviews. 

### Data preperation

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Loading properly with low_memory=False to avoid dtype issues
reviews = pd.read_csv('reviews83325.csv', low_memory=False)
places = pd.read_csv('Tripadvisor.csv', low_memory=False)

reviews_en = reviews[reviews['langue'] == 'en'].copy()

# Combine title + review text for richer signal
reviews_en['full_text'] = reviews_en['titre'].fillna('') + ' ' + reviews_en['review'].fillna('')

# Cap at 50 reviews per place to reduce imbalance
reviews_capped = (
    reviews_en
    .sort_values('note', ascending=False)  # keep best rated reviews
    .groupby('idplace')
    .head(50)
)

# Aggregate all reviews per place into one document
place_docs = (
    reviews_capped
    .groupby('idplace')['full_text']
    .apply(lambda x: ' '.join(x))
    .reset_index()
    .rename(columns={'idplace': 'id', 'full_text': 'doc'})
)

# only keeping places that have English reviews
place_docs = place_docs.merge(places[['id', 'typeR', 'activiteSubCategorie', 
                                       'activiteSubType', 'restaurantType', 
                                       'restaurantTypeCuisine', 'priceRange']], 
                               on='id', how='inner')

print(f"Places with English reviews: {len(place_docs)}")
print(place_docs['typeR'].value_counts())
print(place_docs.head(3))

Places with English reviews: 1835
typeR
AP    989
R     538
A     252
H      56
Name: count, dtype: int64
       id                                                doc typeR  \
0  188467  February Personally I think it is the most bea...     A   
1  188468  one of the best in Paris Walk along the street...     A   
2  188470  A peaceful place to linger in the heart of Par...     A   

  activiteSubCategorie activiteSubType restaurantType restaurantTypeCuisine  \
0                   47             163            NaN                   NaN   
1                   47             163            NaN                   NaN   
2             26,47,51          34,144            NaN                   NaN   

  priceRange  
0        NaN  
1        NaN  
2        NaN  


So, after taking count of places which has english reviews, we have 1835 places with english reviews. 989 Attraction Product, 538 restaurants, 252 attractions, and 56 hotels.

Now we will run 50/50 split.

In [12]:
from sklearn.model_selection import train_test_split
from rank_bm25 import BM25Okapi

# 50/50 split
train_df, test_df = train_test_split(place_docs, test_size=0.5, random_state=42)
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

Train: 917 | Test: 918


Here we will build BM25 index on train set. This is going to be our database.

In [13]:
train_docs_tokenized = [doc.split() for doc in train_df['doc']]
bm25 = BM25Okapi(train_docs_tokenized)

print("BM25 index built successfully!")
print(f"Train typeR distribution:\n{train_df['typeR'].value_counts()}")
print(f"Test typeR distribution:\n{test_df['typeR'].value_counts()}")

BM25 index built successfully!
Train typeR distribution:
typeR
AP    501
R     254
A     128
H      34
Name: count, dtype: int64
Test typeR distribution:
typeR
AP    488
R     284
A     124
H      22
Name: count, dtype: int64


Split looks good as both sets looks to be well balanced, having similar proportions of AP/R/A/H. 

### Level 1 evaluation : ranking error by typeR

In [14]:
import numpy as np

def ranking_error_level1(test_df, train_df, bm25):
    errors = []
    train_ids = train_df['id'].tolist()
    train_types = train_df['typeR'].tolist()
    
    for _, query_row in test_df.iterrows():
        query_type = query_row['typeR']
        query_tokens = query_row['doc'].split()
        
        # Get BM25 scores against all train docs
        scores = bm25.get_scores(query_tokens)
        
        # Rank train places by score (highest first)
        ranked_indices = np.argsort(scores)[::-1]
        
        # Find position of first match (same typeR)
        for rank, idx in enumerate(ranked_indices):
            if train_types[idx] == query_type:
                errors.append(rank)  # 0 = perfect
                break
        # If no match found, skip (don't append)
    
    return errors

errors_l1 = ranking_error_level1(test_df, train_df, bm25)

print(f"Number of queries evaluated: {len(errors_l1)}")
print(f"Mean Ranking Error (Level 1): {np.mean(errors_l1):.4f}")
print(f"Median Ranking Error: {np.median(errors_l1):.4f}")
print(f"Perfect recommendations (error=0): {errors_l1.count(0)} / {len(errors_l1)}")

Number of queries evaluated: 918
Mean Ranking Error (Level 1): 0.7266
Median Ranking Error: 0.0000
Perfect recommendations (error=0): 777 / 918


777/918 means nears 84.6% perfect recomandations of error=0, meaning BM25 found the correct type immediately at rank 1 for most queries. We still have mean error of 0.73 for few errors but having median ranking error at 0 confirms that the most queries are perfect.

## 이 파트 코드 지우기 (verifying)

Verifiying that there is no data leakage.

In [ ]:

print(f"Overlap between train and test: {len(overlap)}")


print(f"BM25 corpus size: {bm25.corpus_size}")  
print(f"Train size: {len(train_df)}")


test_sample = test_df.iloc[0]
print(f"Test place id: {test_sample['id']}")
print(f"Is it in train? {test_sample['id'] in train_ids}")  

Overlap between train and test: 0
BM25 corpus size: 917
Train size: 917
Test place id: 10377984
Is it in train? False


### level 2 evaluation : finding the exact same cuisine or attraction subtype

In [21]:
from tqdm import tqdm
def ranking_error_level2(test_df, train_df, bm25):
    errors = []
    train_types = train_df['typeR'].tolist()
    
    def get_l2_categories(row):
        cats = set()
        if row['typeR'] in ['A', 'AP']:
            if pd.notna(row['activiteSubCategorie']):
                cats.update(str(row['activiteSubCategorie']).split(','))
        elif row['typeR'] == 'R':
            if pd.notna(row['restaurantType']):
                cats.update(str(row['restaurantType']).split(','))
            if pd.notna(row['restaurantTypeCuisine']):
                cats.update(str(row['restaurantTypeCuisine']).split(','))
        elif row['typeR'] == 'H':
            if pd.notna(row['priceRange']):
                cats.add(str(row['priceRange']))
        return cats

    skipped = 0
    for _, query_row in tqdm(test_df.iterrows(), total=len(test_df)):
        query_cats = get_l2_categories(query_row)
        
        # Skip if no metadata available
        if not query_cats:
            skipped += 1
            continue
        
        query_tokens = query_row['doc'].split()
        scores = bm25.get_scores(query_tokens)
        ranked_indices = np.argsort(scores)[::-1]
        
        for rank, idx in enumerate(ranked_indices):
            train_row = train_df.iloc[idx]
            train_cats = get_l2_categories(train_row)
            # Match if at least one category overlaps
            if query_cats & train_cats:
                errors.append(rank)
                break
    
    print(f"Skipped (no metadata): {skipped}")
    return errors

errors_l2 = ranking_error_level2(test_df, train_df, bm25)

print(f"Queries evaluated: {len(errors_l2)}")
print(f"Mean Ranking Error (Level 2): {np.mean(errors_l2):.4f}")
print(f"Median Ranking Error: {np.median(errors_l2):.4f}")

print(f"Perfect recommendations (error=0): {errors_l2.count(0)} / {len(errors_l2)}")

100%|██████████| 918/918 [03:34<00:00,  4.29it/s]

Skipped (no metadata): 489
Queries evaluated: 427
Mean Ranking Error (Level 2): 11.8033
Median Ranking Error: 0.0000
Perfect recommendations (error=0): 316 / 427


489 skipped means nearly half the places had no Level 2 metadata, like having NaN in subcategory, cuisine, priceRange. This is coherent to the number of NaN values that we saw earlier. 
316/427 = 74% perfect with error=0. This is quite good result even though it is lower than Level 1's which was near 85%.
Mean error jumped to 11.8. This proves that BM25 at level 2 fails badly in the ranking to find a subcategory match. 

So the baseline scores that we got are :
Queries evaluated 918 / 427
perfect 84.6% / 74%
mean ranking error 0.73 / 11.8
median ranking error 0 / 0

### 2. Design a model better than BM25

## explanation 다시 내 언어로 쓰기 

Why TF-IDF + Cosine Similarity?
BM25 is a strong baseline but it has a limitation: it scores documents one query term at a time and uses probabilistic weighting. TF-IDF + Cosine Similarity takes a different approach — it represents each place as a vector in a high-dimensional space where each dimension is a word, and measures similarity as the angle between two vectors. This captures the overall vocabulary profile of a place rather than individual term matches, which is more appropriate for our task since we're comparing entire aggregated documents.
Additionally, TF-IDF naturally penalizes very common words (like "the", "good", "nice") that appear everywhere and rewards distinctive words that truly characterize a place — exactly what we need to distinguish between, say, an art museum and a cooking class.

Why add Preprocessing (Model 2)?
Raw text contains a lot of noise — stopwords like "I", "was", "the" carry no meaning, and words like "visiting", "visited", "visit" all mean the same thing but are treated as different words. By adding:

Stopword removal → reduces noise
Stemming/Lemmatization → groups same-meaning words together

We give the TF-IDF vectors a cleaner, more meaningful signal, which should improve similarity matching especially for Level 2 where subcategory distinctions are subtle.

Checking the shapes for train and test.

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

# Build TF-IDF matrix on TRAIN set only
vectorizer = TfidfVectorizer(max_features=10000)
train_tfidf = vectorizer.fit_transform(train_df['doc'])

# Transform test set using the SAME vectorizer (fitted on train only)
test_tfidf = vectorizer.transform(test_df['doc'])

print(f"TF-IDF matrix shape (train): {train_tfidf.shape}")
print(f"TF-IDF matrix shape (test): {test_tfidf.shape}")

TF-IDF matrix shape (train): (917, 10000)
TF-IDF matrix shape (test): (918, 10000)


Level 1 for TF-IDF

In [23]:
def ranking_error_level1_tfidf(test_df, train_df, test_tfidf, train_tfidf):
    errors = []
    train_types = train_df['typeR'].tolist()
    
    # Compute ALL cosine similarities at once (much faster than BM25!)
    similarity_matrix = cosine_similarity(test_tfidf, train_tfidf)
    
    for i, (_, query_row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
        query_type = query_row['typeR']
        scores = similarity_matrix[i]
        ranked_indices = np.argsort(scores)[::-1]
        
        for rank, idx in enumerate(ranked_indices):
            if train_types[idx] == query_type:
                errors.append(rank)
                break
    
    return errors

errors_l1_tfidf = ranking_error_level1_tfidf(test_df, train_df, test_tfidf, train_tfidf)

print(f"Queries evaluated: {len(errors_l1_tfidf)}")
print(f"Mean Ranking Error (Level 1): {np.mean(errors_l1_tfidf):.4f}")
print(f"Median Ranking Error: {np.median(errors_l1_tfidf):.4f}")
print(f"Perfect recommendations (error=0): {errors_l1_tfidf.count(0)} / {len(errors_l1_tfidf)}")

100%|██████████| 918/918 [00:00<00:00, 13875.26it/s]

Queries evaluated: 918
Mean Ranking Error (Level 1): 1.2647
Median Ranking Error: 0.0000
Perfect recommendations (error=0): 797 / 918


We may observe that the run time for this model is much faster than BM25 as it is a type of matrix operations. It does not have to loop through each quert one by one like BM25.
We may observe the result and notice that TF-IDF has more perfect recommendations but higher mean error. This means that when it fails, it fails worse than BM25.

Level 2 for TF-IDF

In [24]:
def ranking_error_level2_tfidf(test_df, train_df, test_tfidf, train_tfidf):
    errors = []
    similarity_matrix = cosine_similarity(test_tfidf, train_tfidf)
    skipped = 0

    def get_l2_categories(row):
        cats = set()
        if row['typeR'] in ['A', 'AP']:
            if pd.notna(row['activiteSubCategorie']):
                cats.update(str(row['activiteSubCategorie']).split(','))
        elif row['typeR'] == 'R':
            if pd.notna(row['restaurantType']):
                cats.update(str(row['restaurantType']).split(','))
            if pd.notna(row['restaurantTypeCuisine']):
                cats.update(str(row['restaurantTypeCuisine']).split(','))
        elif row['typeR'] == 'H':
            if pd.notna(row['priceRange']):
                cats.add(str(row['priceRange']))
        return cats

    for i, (_, query_row) in enumerate(tqdm(test_df.iterrows(), total=len(test_df))):
        query_cats = get_l2_categories(query_row)
        if not query_cats:
            skipped += 1
            continue

        scores = similarity_matrix[i]
        ranked_indices = np.argsort(scores)[::-1]

        for rank, idx in enumerate(ranked_indices):
            train_row = train_df.iloc[idx]
            train_cats = get_l2_categories(train_row)
            if query_cats & train_cats:
                errors.append(rank)
                break

    print(f"Skipped (no metadata): {skipped}")
    return errors

errors_l2_tfidf = ranking_error_level2_tfidf(test_df, train_df, test_tfidf, train_tfidf)

print(f"Queries evaluated: {len(errors_l2_tfidf)}")
print(f"Mean Ranking Error (Level 2): {np.mean(errors_l2_tfidf):.4f}")
print(f"Median Ranking Error: {np.median(errors_l2_tfidf):.4f}")
print(f"Perfect recommendations (error=0): {errors_l2_tfidf.count(0)} / {len(errors_l2_tfidf)}")

100%|██████████| 918/918 [00:00<00:00, 2776.56it/s]

Skipped (no metadata): 489
Queries evaluated: 427
Mean Ranking Error (Level 2): 11.1171
Median Ranking Error: 0.0000
Perfect recommendations (error=0): 329 / 427


The perfect recommendation is 329/427 = 77% which is 3% better than BM25 in level 2. L2 mean error is slightly lower than the BM25 result. BM25 had 11.8 and here we have 11.12.

### Preprocessing + TF-IDF 

In [26]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re

# Download required NLTK data
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove numbers and punctuation
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. Remove stopwords and stem
    tokens = text.split()
    tokens = [stemmer.stem(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

print("Preprocessing train documents...")
train_df_clean = train_df.copy()
test_df_clean = test_df.copy()

train_df_clean['doc_clean'] = train_df_clean['doc'].apply(preprocess)
test_df_clean['doc_clean'] = test_df_clean['doc'].apply(preprocess)

print("Done!")
print("Example before:", train_df['doc'].iloc[0][:100])
print("Example after:", train_df_clean['doc_clean'].iloc[0][:100])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\olivi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


Preprocessing train documents...
Done!
Example before: great gelato very good gelato- this is a toochain as we had some in amboise as well. friendly staff 
Example after: great gelato good gelato toochain ambois well friendli staff linest long hot day bargain compar one 


In [27]:
# Build TF-IDF on clean text
vectorizer_clean = TfidfVectorizer(max_features=10000)
train_tfidf_clean = vectorizer_clean.fit_transform(train_df_clean['doc_clean'])
test_tfidf_clean = vectorizer_clean.transform(test_df_clean['doc_clean'])

print(f"Clean TF-IDF matrix shape (train): {train_tfidf_clean.shape}")
print(f"Clean TF-IDF matrix shape (test): {test_tfidf_clean.shape}")

Clean TF-IDF matrix shape (train): (917, 10000)
Clean TF-IDF matrix shape (test): (918, 10000)


Applying on level 1 

In [28]:
errors_l1_tfidf_clean = ranking_error_level1_tfidf(test_df_clean, train_df_clean, 
                                                     test_tfidf_clean, train_tfidf_clean)

print(f"Queries evaluated: {len(errors_l1_tfidf_clean)}")
print(f"Mean Ranking Error (Level 1): {np.mean(errors_l1_tfidf_clean):.4f}")
print(f"Median Ranking Error: {np.median(errors_l1_tfidf_clean):.4f}")
print(f"Perfect recommendations (error=0): {errors_l1_tfidf_clean.count(0)} / {len(errors_l1_tfidf_clean)}")

100%|██████████| 918/918 [00:00<00:00, 13331.11it/s]

Queries evaluated: 918
Mean Ranking Error (Level 1): 0.6013
Median Ranking Error: 0.0000
Perfect recommendations (error=0): 810 / 918


Better results for model 2 level 1. L1 perfect recommendations score is around 88.2% (810 / 918) which is higher than BM25 and TF-IDF. L1 Mean error is 0.6 and it is lower than BM25 (0.73) and TF-IDF (1.26)

Applying model 2 to level 2 

In [29]:
errors_l2_tfidf_clean = ranking_error_level2_tfidf(test_df_clean, train_df_clean,
                                                     test_tfidf_clean, train_tfidf_clean)

print(f"Queries evaluated: {len(errors_l2_tfidf_clean)}")
print(f"Mean Ranking Error (Level 2): {np.mean(errors_l2_tfidf_clean):.4f}")
print(f"Median Ranking Error: {np.median(errors_l2_tfidf_clean):.4f}")
print(f"Perfect recommendations (error=0): {errors_l2_tfidf_clean.count(0)} / {len(errors_l2_tfidf_clean)}")

100%|██████████| 918/918 [00:00<00:00, 4272.38it/s]

Skipped (no metadata): 489
Queries evaluated: 427
Mean Ranking Error (Level 2): 6.8642
Median Ranking Error: 0.0000
Perfect recommendations (error=0): 329 / 427


L2 Perfect score is same to TF-IDF model which is 77% (329 /427). However, L2 Mean error is significantly lower compare to BM25 (11.8) and TF-IDF (11.12) which is qround 6.86. The preprocessing nearly halved the error at Level 2. This confirms removing stopword and stemming gives TF-IDF a much cleaner signal for finding specific subcategory matches. 